In [2]:
#   3.1 制作数据集
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

# 展示高清图
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')

# 生成数据集
X1 = torch.rand(10000,1)    # 均匀分布
X2 = torch.rand(10000,1)
X3 = torch.rand(10000,1)
# print(X1.shape)
# print(X2.shape)
# print(X3.shape, '\n')

Y1 = ( (X1+X2+X3)<1 ).float()
Y2 = ( (1<(X1+X2+X3)) & ((X1+X2+X3)<2) ).float()
Y3 = ( (X1+X2+X3)>2 ).float()
# print(Y1.shape)
# print(Y2.shape)
# print(Y3.shape,'\n')

Data = torch.cat([X1,X2,X3,Y1,Y2,Y3],axis=1)    # 整合数据集
# Data = Data.to('cuda:0')                      # 把数据集搬到GPU上
# print(Data.shape, '\n')
# print(Data)         # One-Hot编码（独热编码）100-1；010-2；001-3

# 划分训练集与测试集（通用型代码！！！）
train_size = int(0.8*len(Data))     # 训练集的样本数量
test_size = len(Data) - train_size  # 测试集的样本数量
Data = Data[torch.randperm( Data.size(0)) , : ]   # 打乱样本的顺序
train_Data = Data[:train_size, :]   # 训练集样本（0~8000行，所有列）
test_Data = Data[train_size:, :]    # 测试集样本（8000~,所有列）
print(train_Data.size())
print(test_Data.size())

torch.Size([8000, 6])
torch.Size([2000, 6])


In [3]:
#   3.2 搭建神经网络
# __init__特殊方法用于构造自己的神经网络结构，forward方法用于将输入数据进
# 行前向传播。由于张量可以自动计算梯度，所以不需要出现反向传播方法。
class DNN(nn.Module):

    def __init__(self):
        '''搭建神经网络各层'''
        super(DNN, self).__init__()
        self.net = nn.Sequential(           # 按顺序搭建各层
            # 第1层：全连接层，输入维度为 3，输出维度为 5，ReLU作为激活函数
            nn.Linear(3, 5), nn.ReLU(),
            # 第2层：全连接层，输入维度为 5，输出维度为 5
            nn.Linear(5, 5), nn.ReLU(),
            # 第3层：全连接层，输入维度为 5，输出维度为 5
            nn.Linear(5, 5), nn.ReLU(),
            # 第4层：全连接层，输入维度为 5，输出维度为 3
            nn.Linear(5, 3)
        )

    def forward(self, x):
        '''前向传播'''
        y = self.net(x)     # x即输入数据
        return y            # y即输出数据

model = DNN()       # 创建子类的实例
# model = DNN().to('cuda:0')    # 搬到GPU上
print(model)

#     在上面的nn.Sequential()函数中，每一个隐藏层后都使用了ReLU激活函数，
# 各层的神经元节点个数分别是：3、5、5、5、3。
#     注意，输入层有3个神经元、输出层有3个神经元，这不是巧合，是有意而
# 为之。输入层的神经元数量必须与每个样本的输入特征数量一致，输出层的神经
# 元数量必须与每个样本的输出特征数量一致。

DNN(
  (net): Sequential(
    (0): Linear(in_features=3, out_features=5, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5, out_features=5, bias=True)
    (3): ReLU()
    (4): Linear(in_features=5, out_features=5, bias=True)
    (5): ReLU()
    (6): Linear(in_features=5, out_features=3, bias=True)
  )
)


In [4]:
#   3.3 网络的内部参数

# 查看内部参数（非必要）
for name, param in model.named_parameters():
    print(f"参数: {name}\n形状: {param.shape}\n数值:{param}\n")


#   可见，具有权重与偏置的地方只有net.0、net.2、net.4、net.6,结合输出的
# 结果，可知这几个地方其实就是所有的隐藏层与输出层，这符合理论。
#   ●首先，net.0.weight的权重形状为[5,3]，5表示它自己的节点数是5,3表
# 示与之连接的前一层的节点数为3。
#   ●其次，如果进行了model=DNN().to('cuda:0')操作，因此所有的内
# 部参数都自带device='cuda:0'。
#   ●最后，注意到requires_grad=True,说明所有需要进行反向传播的内部参
# 数（即权重与偏置）都打开了张量自带的梯度计算功能。

参数: net.0.weight
形状: torch.Size([5, 3])
数值:Parameter containing:
tensor([[ 0.5060, -0.1287,  0.2211],
        [-0.1512, -0.5322, -0.1322],
        [ 0.1872,  0.1454,  0.1413],
        [ 0.4130, -0.1895,  0.4454],
        [ 0.5097,  0.5626, -0.0216]], requires_grad=True)

参数: net.0.bias
形状: torch.Size([5])
数值:Parameter containing:
tensor([ 0.0596, -0.4273,  0.3622, -0.4604, -0.2663], requires_grad=True)

参数: net.2.weight
形状: torch.Size([5, 5])
数值:Parameter containing:
tensor([[-0.0639,  0.2229,  0.1010,  0.2055, -0.0727],
        [ 0.3323, -0.2351,  0.4067, -0.2980, -0.3397],
        [-0.1595,  0.2710, -0.3620, -0.3542,  0.4374],
        [ 0.3807,  0.2634, -0.2579, -0.3040,  0.2809],
        [-0.1691, -0.0400,  0.2231,  0.2063,  0.1853]], requires_grad=True)

参数: net.2.bias
形状: torch.Size([5])
数值:Parameter containing:
tensor([ 0.4006,  0.2812, -0.1078,  0.0344, -0.0397], requires_grad=True)

参数: net.4.weight
形状: torch.Size([5, 5])
数值:Parameter containing:
tensor([[-0.0509,  0.1492, -0.1

In [5]:
#   3.4 网络的外部参数

# (1)激活函数：https://docs.pytorch.org/docs/2.6/
# 搜索Non-linear Activations，即可查看torch内置的所有非线性激活函数

# (2)损失函数的选择
loss_fn = nn.MSELoss()

# (3)学习率与优化算法:https://pytorch.org/docs/2.6/optim.html
learning_rate = 0.001   # 设置学习率
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 其中的model是3.2中的

In [ ]:
#   3.5 训练网络
epochs = 1000
losses = []     # 记录损失函数变化的列表

# 给训练集划分输入与输出
X = train_Data[:, :3]       # 前3列为输入特征
Y = train_Data[:, -3:]      # 后3列为输出特征

for epoch in range(epochs):
    Pred = model(X)             # 一次前向传播（BGD）
    loss = loss_fn(Pred, Y)     # 计算损失函数
    losses.append(loss.item())  # 记录损失函数的变化(降级为元素，加入列表)
    optimizer.zero_grad()       # 清理上一轮滞留的梯度
    loss.backward()             # 一次反向传播
    optimizer.step()            # 优化内部参数

Fig = plt.figure()
plt.plot(range(epochs), losses)
plt.ylabel('loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
#   3.6 测试网络

# 给测试集划分输入输出
X = train_Data[:, :3]       # 前3列为输入特征
Y = train_Data[:, -3:]      # 后3列为输出特征

with torch.no_grad():       # 关闭梯度计算功能
    Pred =model(X)          # 一次前向传播
# 考虑到输出特征是独热编码，而预测的数据一般都是接近0或1的小数，
# 为了能让预测数据与真实数据之间进行比较，因此要对预测数据进行规整。
    Pred[:,torch.argmax(Pred,axis=1)]=1
    Pred[Pred!=1]=0
#   首先，(Pred==Y)计算预测的输出与真实的输出的各个元素是否相等，返回
# 一个3000行、3列的布尔型张量。
#   其次，(Pred==Y).all(1)检验该布尔型张量每一行的3个数据是否都是True,
# 对于全是True的样本行，结果就是True,否则是False。all(1)中的1表示按“行”
# 扫描，最终返回一个形状为3000的一阶张量。
#   最后，torch.sum((Pred==Y).all(1))的意思就是看这3000个向量相加，True
# 会被当作1，False会被当作0，这样相加刚好就是预测正确的样本数。
    correct = torch.sum((Pred==Y).all(1))   # 预测正确的样本
    total = Y.size(0)           # 第一个维度上的值，即全部的样本数量
    print(f'测试集精准度:{100*correct/total} % ')

In [ ]:
#   3.7 保存与导入网络

# (1)保存网络
torch.save(model, 'model.pth')

# (2)把模型赋给新网络(与model一致，可以直接去跑测试集)
new_model = torch.load('model.pth')

# (3)用新模型进行调试
X = train_Data[:, :3]       # 前3列为输入特征
Y = train_Data[:, -3:]      # 后3列为输出特征

# with torch.no_grad():
#     Pred = new_model(X)
#     Pred[:,torch.argmax(Pred,axis=1)]=1
#     Pred[Pred!=1]=0
#     correct = torch.sum((Pred==Y).all(1))
#     total = Y.size(0)
#     print(f'测试集精准度:{100*correct/total} % ')